# 08 - Phase 2 Data Collection

Collect Phase 2 door/key experiment results. Start with the smoke test to confirm the pipeline, then run the mini experiment cell for usable comparison data.

In [8]:
from pathlib import Path
import importlib
import json
import sys
import time

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np

import src.env.door_key_maze_env as door_key_maze_env
import src.utils.metrics_phase2 as metrics_phase2
import src.training.trainer_phase2 as trainer_phase2

door_key_maze_env = importlib.reload(door_key_maze_env)
metrics_phase2 = importlib.reload(metrics_phase2)
trainer_phase2 = importlib.reload(trainer_phase2)

from src.env import DoorKeyMazeEnv
from src.utils.io import DIRS, ensure_dirs, save_metrics, save_dqn

DoorKeyMazeEnv = door_key_maze_env.DoorKeyMazeEnv
DQN_CFG = trainer_phase2.DQN_CFG
PPO_CFG = trainer_phase2.PPO_CFG
train_dqn_phase2 = trainer_phase2.train_dqn_phase2
train_ppo_phase2 = trainer_phase2.train_ppo_phase2

ensure_dirs()
print("project root:", ROOT)
print("python:", sys.executable)
print("stable_baselines3 available:", importlib.util.find_spec("stable_baselines3") is not None)

project root: C:\Users\pc cam dz\Desktop\RL project
python: c:\anaconda\envs\rlmaze\python.exe
stable_baselines3 available: False


## Default Smoke Settings

In [9]:
SMOKE_TEST = True
RUN_LABEL = "smoke"

MAZE_KWARGS = dict(level="small", difficulty="medium", seed=42, dynamic_objects=True, n_dynamic_layouts=4)
ALGORITHMS = ["dqn", "ddqn", "ppo"]
REWARDS = ["sparse", "subgoal", "potential"]
SEEDS = [0, 1, 2, 3, 4]

ALGORITHMS_TO_RUN = ["ddqn"]
REWARDS_TO_RUN = ["subgoal"]
SEEDS_TO_RUN = [0]
DQN_RUN_CFG = {**DQN_CFG, "episodes": 10, "warmup": 10}
PPO_RUN_CFG = {**PPO_CFG, "total_timesteps": 1024, "n_steps": 256}

print("run label:", RUN_LABEL)
print("algorithms:", ALGORITHMS_TO_RUN)
print("rewards:", REWARDS_TO_RUN)
print("seeds:", SEEDS_TO_RUN)

run label: smoke
algorithms: ['ddqn']
rewards: ['subgoal']
seeds: [0]


## Helpers

In [10]:
def run_id_for(algo, reward, seed):
    mode = globals().get("RUN_LABEL", "smoke" if SMOKE_TEST else "full")
    return f"p2_{mode}_{algo}_{reward}_{MAZE_KWARGS['level']}_{MAZE_KWARGS['difficulty']}_seed{seed}"


def tracker_optimal_path_arr(tracker):
    if hasattr(tracker, "optimal_path_arr"):
        return tracker.optimal_path_arr
    return np.full_like(tracker.steps_arr, tracker.optimal_path_len, dtype=float)


def save_phase2_result(run_id, env, result, algo, reward, seed):
    tracker = result.tracker
    metrics_path = save_metrics({
        "episodes": tracker.episodes_arr,
        "rewards": tracker.rewards,
        "steps": tracker.steps_arr,
        "success": tracker.success_arr,
        "key": tracker.key_arr,
        "door": tracker.door_arr,
        "steps_to_key": tracker.steps_to_key_arr,
        "steps_to_door": tracker.steps_to_door_arr,
        "optimal_path": tracker_optimal_path_arr(tracker),
    }, run_id)
    if algo in {"dqn", "ddqn"}:
        save_dqn(result.model, run_id)

    summary = tracker.compute_summary()
    summary.update({
        "run_id": run_id,
        "algo": algo,
        "reward": reward,
        "seed": seed,
        "maze_level": env.level,
        "maze_size": env.size,
        "difficulty": env.difficulty,
        "key_pos": list(env.key_pos),
        "door_pos": list(env.door_pos),
        "dynamic_objects": env.dynamic_objects,
        "n_dynamic_layouts": len(env._layouts),
        "metrics_path": str(metrics_path),
    })
    summary_path = DIRS["logs"] / f"{run_id}_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary


def evaluate_ppo_policy(env, model, episodes=100):
    rewards, steps, success, key, door, steps_to_key, steps_to_door, optimal_path = [], [], [], [], [], [], [], []
    for _ in range(episodes):
        obs, _ = env.reset()
        total = 0.0
        done = False
        info = {}
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(int(action))
            total += float(reward)
            done = terminated or truncated
        rewards.append(total)
        steps.append(env.steps_taken)
        success.append(int(terminated))
        key.append(info.get("picked_up_key", 0))
        door.append(info.get("opened_door", 0))
        steps_to_key.append(np.nan if info.get("steps_to_key") is None else info.get("steps_to_key"))
        steps_to_door.append(np.nan if info.get("steps_to_door") is None else info.get("steps_to_door"))
        optimal_path.append(info.get("optimal_path_len", np.nan))
    return {
        "episodes": np.arange(1, episodes + 1),
        "rewards": np.array(rewards, dtype=float),
        "steps": np.array(steps, dtype=float),
        "success": np.array(success, dtype=float),
        "key": np.array(key, dtype=float),
        "door": np.array(door, dtype=float),
        "steps_to_key": np.array(steps_to_key, dtype=float),
        "steps_to_door": np.array(steps_to_door, dtype=float),
        "optimal_path": np.array(optimal_path, dtype=float),
    }


def save_ppo_summary(run_id, env, model, key_log, door_log, algo, reward, seed):
    eval_episodes = 5 if SMOKE_TEST else 100
    eval_metrics = evaluate_ppo_policy(env, model, episodes=eval_episodes)
    metrics_path = save_metrics(eval_metrics, run_id)
    summary = {
        "run_id": run_id,
        "algo": algo,
        "reward": reward,
        "seed": seed,
        "maze_level": env.level,
        "maze_size": env.size,
        "difficulty": env.difficulty,
        "key_pos": list(env.key_pos),
        "door_pos": list(env.door_pos),
        "dynamic_objects": env.dynamic_objects,
        "n_dynamic_layouts": len(env._layouts),
        "success_rate": float(np.mean(eval_metrics["success"])),
        "key_pickup_rate": float(np.mean(eval_metrics["key"])),
        "door_opening_rate": float(np.mean(eval_metrics["door"])),
        "key_pickup_rate_logged": float(np.mean(key_log)) if key_log else None,
        "door_opening_rate_logged": float(np.mean(door_log)) if door_log else None,
        "metrics_path": str(metrics_path),
    }
    path = DIRS["logs"] / f"{run_id}_summary.json"
    path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary


def print_summary_table(summaries):
    cols = [
        "algo", "reward", "seed", "success_rate", "key_pickup_rate",
        "door_opening_rate", "mean_steps_to_goal", "mean_steps_to_key",
        "mean_key_to_door_steps", "path_optimality",
    ]
    if not summaries:
        print("No summaries were produced.")
        return
    widths = {col: max(len(col), *(len(str(s.get(col))) for s in summaries)) for col in cols}
    print(" | ".join(col.ljust(widths[col]) for col in cols))
    print("-+-".join("-" * widths[col] for col in cols))
    for s in summaries:
        print(" | ".join(str(s.get(col)).ljust(widths[col]) for col in cols))


def run_phase2_experiment_matrix():
    summaries = []
    start_time = time.time()
    total_runs = len(ALGORITHMS_TO_RUN) * len(REWARDS_TO_RUN) * len(SEEDS_TO_RUN)
    print("runs:", total_runs)
    print("algorithms:", ALGORITHMS_TO_RUN)
    print("rewards:", REWARDS_TO_RUN)
    print("seeds:", SEEDS_TO_RUN)

    for algo in ALGORITHMS_TO_RUN:
        for reward in REWARDS_TO_RUN:
            for seed in SEEDS_TO_RUN:
                run_id = run_id_for(algo, reward, seed)
                env = DoorKeyMazeEnv(**MAZE_KWARGS, reward_type=reward)
                print("\nRUN", run_id)
                print("key=", env.key_pos, "door=", env.door_pos, "optimal=", env.optimal_path_length())

                if algo == "dqn":
                    result = train_dqn_phase2(env, cfg=DQN_RUN_CFG, seed=seed, double=False)
                    summary = save_phase2_result(run_id, env, result, algo, reward, seed)
                elif algo == "ddqn":
                    result = train_dqn_phase2(env, cfg=DQN_RUN_CFG, seed=seed, double=True)
                    summary = save_phase2_result(run_id, env, result, algo, reward, seed)
                elif algo == "ppo":
                    model, key_log, door_log = train_ppo_phase2(env, cfg=PPO_RUN_CFG, seed=seed)
                    summary = save_ppo_summary(run_id, env, model, key_log, door_log, algo, reward, seed)
                else:
                    raise ValueError(algo)

                summaries.append(summary)
                print(summary)

    print("\nelapsed seconds:", round(time.time() - start_time, 1))
    print("\nsummary table:")
    print_summary_table(summaries)
    return summaries

**Experiments:**

## Trial 1: DDQN Sparse

In [ ]:
SMOKE_TEST = True
RUN_LABEL = "quick"

ALGORITHMS_TO_RUN = ["ddqn"]
REWARDS_TO_RUN = ["sparse"]
SEEDS_TO_RUN = [0]

DQN_RUN_CFG = {**DQN_CFG, "episodes": 1000, "warmup": 500}
PPO_RUN_CFG = PPO_CFG

ddqn_sparse_summary = run_phase2_experiment_matrix()

## Trial 2: DDQN Potential

In [ ]:
SMOKE_TEST = True
RUN_LABEL = "quick"

ALGORITHMS_TO_RUN = ["ddqn"]
REWARDS_TO_RUN = ["potential"]
SEEDS_TO_RUN = [0]

DQN_RUN_CFG = {**DQN_CFG, "episodes": 1000, "warmup": 500}
PPO_RUN_CFG = PPO_CFG

ddqn_potential_summary = run_phase2_experiment_matrix()

Trial 3: DQN Subgoal

In [ ]:
SMOKE_TEST = True
RUN_LABEL = "quick"

ALGORITHMS_TO_RUN = ["dqn"]
REWARDS_TO_RUN = ["subgoal"]
SEEDS_TO_RUN = [0]

DQN_RUN_CFG = {**DQN_CFG, "episodes": 1000, "warmup": 500}
PPO_RUN_CFG = PPO_CFG

dqn_subgoal_summary = run_phase2_experiment_matrix()

## Trial 4: PPO Subgoal

In [13]:
%pip install stable-baselines3

   ---------------------------------------- 0.0/952.1 kB ? eta -:--:--
   ---------------------------------------- 952.1/952.1 kB 6.3 MB/s  0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------- ----------------- 5.5/9.9 MB 25.8 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.9 MB 25.8 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 16.7 MB/s  0:00:00

  Attempting uninstall: gymnasium

    Found existing installation: gymnasium 1.3.0

    Uninstalling gymnasium-1.3.0:

      Successfully uninstalled gymnasium-1.3.0

   ---------------------------------------- 0/3 [gymnasium]
   ---------------------------------------- 0/3 [gymnasium]
   ---------------------------------------- 0/3 [gymnasium]
   ---------------------------------------- 0/3 [gymnasium]
   ---------------------------------------- 0/3 [gymnasium]
   ---------------------------------------- 0/3 [gymnasium]
   -----------------------

In [14]:
SMOKE_TEST = True
RUN_LABEL = "quick"

ALGORITHMS_TO_RUN = ["ppo"]
REWARDS_TO_RUN = ["subgoal"]
SEEDS_TO_RUN = [0]

DQN_RUN_CFG = DQN_CFG
PPO_RUN_CFG = {**PPO_CFG, "total_timesteps": 100_000}

if importlib.util.find_spec("stable_baselines3") is None:
    print("PPO skipped: stable_baselines3 is not installed in this notebook kernel.")
    print("Notebook python:", sys.executable)
    print("Fix: select the Python/kernel where stable-baselines3 is installed, or install it in this kernel.")
    print("Command inside this notebook kernel: %pip install stable-baselines3")
    ppo_subgoal_summary = []
else:
    ppo_subgoal_summary = run_phase2_experiment_matrix()

runs: 1
algorithms: ['ppo']
rewards: ['subgoal']
seeds: [0]

RUN p2_quick_ppo_subgoal_small_medium_seed0
key= (9, 6) door= (2, 5) optimal= 52
  [io] metrics saved -> C:\Users\pc cam dz\Desktop\RL project\results\metrics\p2_quick_ppo_subgoal_small_medium_seed0_metrics.npz
{'run_id': 'p2_quick_ppo_subgoal_small_medium_seed0', 'algo': 'ppo', 'reward': 'subgoal', 'seed': 0, 'maze_level': 'small', 'maze_size': 10, 'difficulty': 'medium', 'key_pos': [0, 9], 'door_pos': [7, 2], 'dynamic_objects': True, 'n_dynamic_layouts': 4, 'success_rate': 0.0, 'key_pickup_rate': 0.0, 'door_opening_rate': 0.0, 'key_pickup_rate_logged': 0.021259765625, 'door_opening_rate_logged': 0.00275390625, 'metrics_path': 'C:\\Users\\pc cam dz\\Desktop\\RL project\\results\\metrics\\p2_quick_ppo_subgoal_small_medium_seed0_metrics.npz'}

elapsed seconds: 172.3

summary table:
algo | reward  | seed | success_rate | key_pickup_rate | door_opening_rate | mean_steps_to_goal | mean_steps_to_key | mean_key_to_door_steps | path

## Smoke Run

In [ ]:
SMOKE_TEST = True
RUN_LABEL = "smoke"

ALGORITHMS_TO_RUN = ["ddqn"]
REWARDS_TO_RUN = ["subgoal"]
SEEDS_TO_RUN = [0]

DQN_RUN_CFG = {**DQN_CFG, "episodes": 10, "warmup": 10}
PPO_RUN_CFG = {**PPO_CFG, "total_timesteps": 1024, "n_steps": 256}

summaries = run_phase2_experiment_matrix()

## Existing Quick DDQN Subgoal Run

In [ ]:
SMOKE_TEST = True
RUN_LABEL = "quick"

ALGORITHMS_TO_RUN = ["ddqn"]
REWARDS_TO_RUN = ["subgoal"]
SEEDS_TO_RUN = [0]

DQN_RUN_CFG = {**DQN_CFG, "episodes": 1000, "warmup": 500}
PPO_RUN_CFG = PPO_CFG

quick_summaries = run_phase2_experiment_matrix()

## Verify Quick Comparison Coverage

In [15]:
expected_quick = [
    ("ddqn", "sparse"),
    ("ddqn", "subgoal"),
    ("ddqn", "potential"),
    ("dqn", "subgoal"),
    ("ppo", "subgoal"),
]

existing = set()
for path in sorted(DIRS["metrics"].glob("p2_quick_*_metrics.npz")):
    run_id = path.name.replace("_metrics.npz", "")
    parts = run_id.split("_")
    existing.add((parts[2], parts[3]))

print("existing quick runs:", sorted(existing))
missing = [pair for pair in expected_quick if pair not in existing]
print("missing quick runs:", missing if missing else "none")

if missing:
    print("Run the trial cells above for the missing pairs before notebooks 10-12.")
else:
    print("Quick comparison set is ready. Move to notebooks 09-12.")

existing quick runs: [('ddqn', 'potential'), ('ddqn', 'sparse'), ('ddqn', 'subgoal'), ('dqn', 'subgoal'), ('ppo', 'subgoal')]
missing quick runs: none
Quick comparison set is ready. Move to notebooks 09-12.


## Inspect Saved Summaries

In [16]:
summary_files = sorted(DIRS["logs"].glob("p2_*_summary.json"))
print("Phase 2 summary files:", len(summary_files))
for path in summary_files[-10:]:
    print(path.name)

Phase 2 summary files: 6
p2_quick_ddqn_potential_small_medium_seed0_summary.json
p2_quick_ddqn_sparse_small_medium_seed0_summary.json
p2_quick_ddqn_subgoal_small_medium_seed0_summary.json
p2_quick_dqn_subgoal_small_medium_seed0_summary.json
p2_quick_ppo_subgoal_small_medium_seed0_summary.json
p2_smoke_ddqn_subgoal_small_medium_seed0_summary.json
